# Phase 2 — Classical Machine Learning
**Models:** K-Nearest Neighbours · Logistic Regression · Random Forest · XGBoost  
**Goal:** Train, evaluate, and compare four classical ML classifiers on the preprocessed CICIDS2017 dataset.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib, os, warnings
warnings.filterwarnings('ignore')

DATA_PATH   = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data"
MODELS_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\models"
PLOTS_PATH  = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\reports"
os.makedirs(MODELS_PATH, exist_ok=True)

## 2. Load Data

In [ ]:
df = pd.read_parquet(os.path.join(DATA_PATH, 'clean_data.parquet'))
le = joblib.load(os.path.join(DATA_PATH, 'label_encoder.pkl'))

X = df.drop('Label', axis=1)
y = df['Label']
print(f"✅ {df.shape[0]:,} rows  ·  {X.shape[1]} features  ·  {len(le.classes_)} classes")

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

## 4. Class Balancing with SMOTE

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
sampling_dict = {u: 10000 for u, c in zip(unique, counts) if c < 10000}

smote = SMOTE(sampling_strategy=sampling_dict, random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Before SMOTE : {X_train.shape[0]:,}  |  After SMOTE : {X_train_sm.shape[0]:,}")

## 5. Evaluation Helper

In [ ]:
def evaluate_model(name, model, X_test, y_test, le):
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"{'='*50}\n  {name}\n{'='*50}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}  |  Recall : {rec:.4f}  |  F1 : {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(14, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix — {name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_PATH, f'cm_{name.replace(" ", "_")}.png'), dpi=150)
    plt.show()
    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

## 6. Model Training & Evaluation

### 6.1 K-Nearest Neighbours

In [ ]:
sample_idx  = np.random.choice(len(X_train_sm), 100000, replace=False)
X_train_knn = X_train_sm.iloc[sample_idx]
y_train_knn = y_train_sm.iloc[sample_idx]

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train_knn, y_train_knn)
results_knn = evaluate_model("KNN", knn, X_test, y_test, le)
joblib.dump(knn, os.path.join(MODELS_PATH, 'knn_model.pkl'))
print("✅ KNN model saved")

### 6.2 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, C=1.0)
lr.fit(X_train_sm, y_train_sm)
results_lr = evaluate_model("Logistic Regression", lr, X_test, y_test, le)
joblib.dump(lr, os.path.join(MODELS_PATH, 'lr_model.pkl'))
print("✅ Logistic Regression model saved")

### 6.3 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, max_depth=20, min_samples_split=10,
    random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)
results_rf = evaluate_model("Random Forest", rf, X_test, y_test, le)
joblib.dump(rf, os.path.join(MODELS_PATH, 'rf_model.pkl'))
print("✅ Random Forest model saved")

### 6.4 XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42,
    n_jobs=-1, tree_method='hist')
xgb.fit(X_train_sm, y_train_sm, eval_set=[(X_test, y_test)], verbose=50)
results_xgb = evaluate_model("XGBoost", xgb, X_test, y_test, le)
joblib.dump(xgb, os.path.join(MODELS_PATH, 'xgb_model.pkl'))
print("✅ XGBoost model saved")

## 7. Model Comparison

In [ ]:
results = [results_knn, results_lr, results_rf, results_xgb]
results_df = pd.DataFrame(results)

metrics = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(results_df))
width = 0.2
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 7))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i * width, results_df[metric], width,
           label=metric.capitalize(), color=color, alpha=0.85)

ax.set_xlabel('Model', fontsize=13); ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['name'], fontsize=11)
ax.set_ylim(0.95, 1.005); ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)

for i, metric in enumerate(metrics):
    for j, val in enumerate(results_df[metric]):
        ax.text(j + i * width, val + 0.0003, f'{val:.4f}',
                ha='center', va='bottom', fontsize=7, rotation=90)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_PATH, 'model_comparison.png'), dpi=150)
plt.show()

best = results_df.loc[results_df['f1'].idxmax()]
print(f"🏆 Best model : {best['name']}  (F1 = {best['f1']:.4f})")
results_df.to_csv(os.path.join(PLOTS_PATH, 'model_results.csv'), index=False)